# RG3 가스 → 기존 최고 F1 미성형 후보

## 이번 조합

- 가스: 기존 최고 개발 후보, 17개 중 8개 탐지·정상 오탐 6개·F1 0.516. 노트북 152에서 전체 RG3 591샷에 실제 적용한 구간제외 예측을 재사용한다.
- 미성형: 노트북 143의 고정 A OR B OR C OR D 후보, 8개 중 6개 탐지·정상 오탐 11개·F1 0.480. 최근 공통 이력 후보 F1 0.435와 구별한다.
- A: 센서 std24 기반 불량 유형 전문가(SVM/로지스틱 계열). B: 운전조건을 고려한 관계 피처 앙상블. C: 온도·쿠션 변화 앙상블에 배럴5 흔들림 확인 조건. D: 반대 방향 동시변화 강도 로지스틱에 호퍼 냉각 확인 조건. 상세 원래 정의·모델·임계값은 133/139/141/143 노트북에 보존된다.

## 계산과 해석

공식 기존 RG3 RH 591샷(정상566·가스17·미성형8)을 사용한다. 가스로 예측한 샷을 먼저 제거하고, 남은 샷에서 미성형을 판정한다. 정답으로 분기하지 않는다. 센서 과거 이력도 제거 후 다시 계산하지 않는다.

기존에 계산한 구간제외 예측을 결합하므로 이 노트북은 학습 노트북이 아니라 **고정 후보 결합·지표 계산 노트북**이다. 각 평가 구간을 학습에서 제외한 예측이지만 반복 개발 과정에서 선택한 후보이므로 미관측 최종 독립검증 성적은 아니다.

기존 최고 미성형 후보는 일부 피처의 이력 계산 방식이 최근 공통 이력 후보와 다르다. 라벨 자료에서 최고 F1이라는 뜻이며, 비라벨 운전조건에서도 가장 좋다는 뜻은 아니다. 비라벨 적용성 문제를 해결했다고 주장하지 않는다.

단계별 지표·최종 불량 여부·3종류 혼동행렬·미성형 8개 전체 판정·정상 오탐 목록·이전 조합 비교를 저장한다. 새로운 튜닝 또는 커밋/푸시는 하지 않는다.

In [1]:
from pathlib import Path
import hashlib,json
import numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix
root=Path.cwd();t=root/'reports/tables'
# Frozen, previously generated held-out-block predictions: no retraining,
# threshold search, or selection using this cascade's evaluation labels.
gp=t/'rg3_152_gas_shortshot_cascade_predictions.csv'
sp=t/'rg3_shortshot_143_fixed_candidate_predictions.csv'
gas=pd.read_csv(gp,parse_dates=['TimeStamp'])
short=pd.read_csv(sp,parse_dates=['TimeStamp'])
assert len(gas)==len(short)==591 and gas.TimeStamp.is_unique and short.TimeStamp.is_unique
short=short[['TimeStamp','run_id','type','prediction','A','B','C','D']].rename(columns={'prediction':'shortshot_prediction','type':'shortshot_actual_type'})
p=gas[['TimeStamp','block','type','gas_prediction']].merge(short,on='TimeStamp',validate='one_to_one')
assert p.type.eq(p.shortshot_actual_type).all()
assert p.type.value_counts().to_dict()=={'정상':566,'가스':17,'미성형':8}
mapping=p.groupby('block').run_id.nunique()
assert mapping.eq(1).all() and p.groupby('run_id').block.nunique().eq(1).all()
assert p.shortshot_prediction.eq(p[['A','B','C','D']].astype(bool).any(axis=1).astype(int)).all()
p['stage2_checked']=p.gas_prediction.eq(0)
p['stage2_flag']=p.stage2_checked&p.shortshot_prediction.eq(1)
p['final_defect']=p.gas_prediction.eq(1)|p.stage2_flag
p['predicted_type']=np.where(p.gas_prediction.eq(1),'가스',np.where(p.stage2_flag,'미성형','정상'))
p=p.sort_values('TimeStamp').reset_index(drop=True)
# Features remain based on the pre-filter sensor history; routing uses ONLY predictions.
def metric(name,truth,pred):
 tn,fp,fn,tp=confusion_matrix(truth,pred,labels=[0,1]).ravel()
 return dict(evaluation=name,shots=len(truth),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),precision=tp/(tp+fp) if tp+fp else 0,recall=tp/(tp+fn) if tp+fn else 0,F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0)
ng=p.type.isin(['정상','가스']);ns=p.type.isin(['정상','미성형'])
m=pd.DataFrame([
 metric('gas_only_583',p.loc[ng,'type'].eq('가스'),p.loc[ng,'gas_prediction']),
 metric('shortshot_best_alone_574',p.loc[ns,'type'].eq('미성형'),p.loc[ns,'shortshot_prediction']),
 metric('final_any_defect_591',p.type.ne('정상'),p.final_defect),
 metric('gas_type_full_591',p.type.eq('가스'),p.predicted_type.eq('가스')),
 metric('shortshot_type_full_591',p.type.eq('미성형'),p.predicted_type.eq('미성형'))])
assert m.loc[m.evaluation.eq('gas_only_583'),['TP','FP','FN','TN']].values.tolist()==[[8,6,9,560]]
assert m.loc[m.evaluation.eq('shortshot_best_alone_574'),['TP','FP','FN','TN']].values.tolist()==[[6,11,2,555]]
matrix=pd.crosstab(p.type,p.predicted_type).reindex(index=['정상','가스','미성형'],columns=['정상','가스','미성형'],fill_value=0)
steps=[]
for name,flag in [('gas_stage',p.gas_prediction.eq(1)),('shortshot_stage_remaining_only',p.stage2_flag)]:
 steps.append(dict(stage=name,flagged=int(flag.sum()),true_gas=int((flag&p.type.eq('가스')).sum()),true_shortshot=int((flag&p.type.eq('미성형')).sum()),normal_false_alarm=int((flag&p.type.eq('정상')).sum())))
previous=pd.read_csv(t/'rg3_152_gas_shortshot_cascade_metrics.csv')
comparison=pd.concat([previous.loc[previous.evaluation.eq('final_any_defect_591')].assign(candidate='gas + shared-history shortshot (151)'),m.loc[m.evaluation.eq('final_any_defect_591')].assign(candidate='gas + highest-F1 shortshot (143)')],ignore_index=True)
byblock=[]
for block,q in p.groupby('block'):
 byblock.append(dict(block=block,**metric('any_defect',q.type.ne('정상'),q.final_defect)))
for obj,name in [(p,'predictions'),(m,'metrics'),(p.loc[p.type.ne('정상')],'all_defects'),(p.loc[p.type.eq('미성형')],'all8_shortshots'),(p.loc[p.type.eq('정상')&p.final_defect],'false_positives'),(pd.DataFrame(steps),'stage_counts'),(comparison,'comparison'),(pd.DataFrame(byblock),'byblock')]:
 obj.to_csv(t/f'rg3_153_gas_best_shortshot_cascade_{name}.csv',index=False,encoding='utf-8-sig')
matrix.to_csv(t/'rg3_153_gas_best_shortshot_cascade_type_matrix.csv',encoding='utf-8-sig')
manifest={'evaluation':'frozen development held-out-block predictions, not unseen final test','gas_source':str(gp),'shortshot_source':str(sp),'gas_sha256':hashlib.sha256(gp.read_bytes()).hexdigest(),'shortshot_sha256':hashlib.sha256(sp.read_bytes()).hexdigest(),'cohort':{'normal':566,'gas':17,'shortshot':8},'shortshot_candidate':'fixed A|B|C|D, notebook 143; historical mixed sensor-history definitions; unlabeled transport not established','stage_routing':'gas prediction first; shortshot prediction only if gas negative'}
(t/'rg3_153_gas_best_shortshot_cascade_provenance.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print(m.to_string(index=False))
print('\nSTAGES');print(pd.DataFrame(steps).to_string(index=False))
print('\nTYPE MATRIX (actual rows / predicted columns)');print(matrix.to_string())
print('\nCOMPARISON');print(comparison.to_string(index=False))
print('\nALL EIGHT SHORTSHOTS');print(p.loc[p.type.eq('미성형'),['TimeStamp','A','B','C','D','gas_prediction','shortshot_prediction','predicted_type']].to_string(index=False))


              evaluation  shots  TP  FP  FN  TN  precision   recall       F1
            gas_only_583    583   8   6   9 560   0.571429 0.470588 0.516129
shortshot_best_alone_574    574   6  11   2 555   0.352941 0.750000 0.480000
    final_any_defect_591    591  14  17  11 549   0.451613 0.560000 0.500000
       gas_type_full_591    591   8   6   9 568   0.571429 0.470588 0.516129
 shortshot_type_full_591    591   6  11   2 572   0.352941 0.750000 0.480000

STAGES
                         stage  flagged  true_gas  true_shortshot  normal_false_alarm
                     gas_stage       14         8               0                   6
shortshot_stage_remaining_only       17         0               6                  11

TYPE MATRIX (actual rows / predicted columns)
predicted_type   정상  가스  미성형
type                        
정상              549   6   11
가스                9   8    0
미성형               2   0    6

COMPARISON
          evaluation  shots  TP  FP  FN  TN  precision  recall      